[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/42_rloo_loss.ipynb)

# 🔴 困难：留一法策略梯度损失（RLOO）

实现**留一法策略梯度（REINFORCE Leave-One-Out，RLOO）** 损失：这是一种无需价值网络（critic-free）、无偏且低方差的策略梯度目标，常用于大语言模型的 RLHF 与 RLAIF 对齐训练。

对每个提示词（对应一个组 $g$）采样 $K_g$ 条回复。当 $K_g \ge 2$ 时，样本 $i$ 的**留一基线（Leave-One-Out Baseline）** 定义为同组内**其余** $K_g - 1$ 个样本奖励的均值：

$$
b_i = \frac{1}{K_{g(i)} - 1} \sum_{\substack{j \in g(i) \\ j \neq i}} r_j
$$

对应的 RLOO 优势为：

$$
A_i = r_i - b_i = \frac{K_{g(i)}}{K_{g(i)} - 1}\big(r_i - \bar r_{g(i)}\big)
$$

注意与 GRPO 和普通均值基线的区别：
- **无偏基线**：由于 $b_i$ 不依赖当前样本 $y_i$，$A_i \nabla_\theta \log \pi_\theta(y_i \mid x)$ 是无偏策略梯度估计（普通组均值基线 $r_i - \bar r_g$ 会引入 $\frac{K_g - 1}{K_g}$ 的缩放偏差）。
- **不除以组内标准差**：与 GRPO 不同，RLOO **不**除以 $\text{std}_g$，保留奖励差值的真实尺度。
- **单样本组约定**：当某个组仅有 $K_g = 1$ 个样本时，无法构造留一基线，约定其优势 $A_i = 0$。

RLOO 将整条回复视为单个动作：若传入的 `logps` 为词元级形状 $(B, T)$，需先在词元维度上对有效位置**求和**（如提供 `mask`，仅计入 `mask` 为 $1$ 的词元）得到序列级对数概率 $\log \pi_\theta(y_i \mid x)$，再计算负均值损失：

$$
\mathcal{L}_{\text{RLOO}} = -\mathbb{E}_i \big[\,\text{stop\_grad}(A_i)\, \log \pi_\theta(y_i \mid x)\big].
$$

### 函数或类接口
```python
from torch import Tensor

def rloo_loss(logps: Tensor,
              rewards: Tensor,
              group_ids: Tensor,
              mask: Tensor | None = None) -> Tensor:
    """计算一个批次的 RLOO 策略梯度损失。

    logps: (B,) 序列级对数概率，或 (B, T) 词元级对数概率
    rewards: (B,) 每个采样回复的标量奖励
    group_ids: (B,) 整数组编号，相同编号表示同一个提示词／组（不一定连续或等长）
    mask: 可选，形状为 (B, T) 的词元掩码，1 表示计入，0 表示忽略
    返回：标量损失（Tensor），梯度仅通过 logps 传播
    """
```

In [ ]:
# 在 Colab 中安装 torch-judge（JupyterLab/Docker 中会跳过）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn.functional as F
from torch import Tensor

In [ ]:
# ✏️ 在此编写你的实现

from torch import Tensor

def rloo_loss(logps: Tensor,
              rewards: Tensor,
              group_ids: Tensor,
              mask: Tensor | None = None) -> Tensor:
    pass  # 按组计算留一基线优势 A_i = r_i - b_i，返回 -mean(A_i.detach() * seq_logps)

In [ ]:
# 🧪 调试
logps = torch.tensor([-0.5, -1.5, -0.2, -0.8], requires_grad=True)
rewards = torch.tensor([1.0, 0.0, 2.0, 0.5])
group_ids = torch.tensor([0, 0, 1, 1])
print('Loss:', rloo_loss(logps, rewards, group_ids).item())

In [ ]:
# ✅ 提交
from torch_judge import check
check('rloo_loss')